In [1]:
import pandas as pd

In [4]:
from pathlib import Path

raw_folder = Path("../data/raw")

list(raw_folder.iterdir())

[WindowsPath('../data/raw/complaints-2026-10-06_12_02.csv')]

In [5]:
file_path = '../data/raw/complaints-2026-10-06_12_02.csv'

df = pd.read_csv(file_path)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Rows: 60,553
Columns: 15


In [ ]:
# Checking the fields we have

df.columns.tolist()

['Date received',
 'Product',
 'Sub-product',
 'Issue',
 'Sub-issue',
 'Company public response',
 'Company',
 'State',
 'ZIP code',
 'Tags',
 'Submitted via',
 'Date sent to company',
 'Company response to consumer',
 'Timely response?',
 'Complaint ID']

In [7]:
# Checking the Data Types

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60553 entries, 0 to 60552
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   Date received                 60553 non-null  str  
 1   Product                       60553 non-null  str  
 2   Sub-product                   60553 non-null  str  
 3   Issue                         60553 non-null  str  
 4   Sub-issue                     4596 non-null   str  
 5   Company public response       3544 non-null   str  
 6   Company                       60553 non-null  str  
 7   State                         60139 non-null  str  
 8   ZIP code                      60553 non-null  str  
 9   Tags                          3554 non-null   str  
 10  Submitted via                 60553 non-null  str  
 11  Date sent to company          60553 non-null  str  
 12  Company response to consumer  60553 non-null  str  
 13  Timely response?              60553 non-nu

In [8]:
# Quantifying the Missing Data

missing = df.isna().sum()

missing_percentage = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing,
    "Missing %": missing_percentage
})

missing_summary.sort_values("Missing Values", ascending=False)

,Missing Values,Missing %
Company public response,57009,94.15
Tags,56999,94.13
Sub-issue,55957,92.41
State,414,0.68
Product,0,0.00
Sub-product,0,0.00
Date received,0,0.00
Company,0,0.00
Issue,0,0.00
ZIP code,0,0.00


In [9]:
#Checking for Duplicate Complaint

duplicate_count = df["Complaint ID"].duplicated().sum()

print(f"Duplicate Complaint IDs: {duplicate_count:,}")

Duplicate Complaint IDs: 0


In [10]:
date_columns = [
    "Date received",
    "Date sent to company"
]

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

df[date_columns].dtypes

Date received           datetime64[us, UTC]
Date sent to company    datetime64[us, UTC]
dtype: object

In [11]:
for column in date_columns:
    print(
        f"{column}: "
        f"{df[column].isna().sum():,} missing dates"
    )

Date received: 0 missing dates
Date sent to company: 0 missing dates


In [12]:
print("Earliest complaint:", df["Date received"].min())
print("Latest complaint:", df["Date received"].max())

print("\nEarliest sent to company:", df["Date sent to company"].min())
print("Latest sent to company:", df["Date sent to company"].max())

Earliest complaint: 2025-01-01 03:51:42+00:00
Latest complaint: 2025-12-31 23:57:28+00:00

Earliest sent to company: 2025-01-01 04:11:27+00:00
Latest sent to company: 2026-06-11 13:44:26+00:00


In [13]:
#What are customers really complaining about?

issue_summary = (
    df["Issue"]
    .value_counts()
    .reset_index()
)

issue_summary.columns = ["Issue", "Complaints"]

issue_summary

,Issue,Complaints
0,Other transaction problem,53893
1,Problem when making payments,4596
2,Other service problem,2064


In [14]:
# Which customer products are generating the greatest volume of these problems?

product_summary = (
    df["Product"]
    .value_counts()
    .reset_index()
)

product_summary.columns = ["Product", "Complaints"]

product_summary

,Product,Complaints
0,"Money transfer, virtual currency, or money ser...",55957
1,Credit card,4596


In [15]:
# Product x Customer Problem

product_issue = pd.crosstab(
    df["Product"],
    df["Issue"]
)

product_issue

Issue,Other service problem,Other transaction problem,Problem when making payments
Product,,,
Credit card,0,0,4596
"Money transfer, virtual currency, or money service",2064,53893,0


In [16]:
response_summary = (
    df["Company response to consumer"]
    .value_counts()
    .reset_index()
)

response_summary.columns = ["Company Response", "Complaints"]

response_summary

,Company Response,Complaints
0,Closed with explanation,58564
1,Closed with monetary relief,1337
2,Closed with non-monetary relief,609
3,Untimely response,42
4,In progress,1


In [17]:
timely_summary = (
    df["Timely response?"]
    .value_counts(dropna=False)
    .reset_index()
)

timely_summary.columns = ["Timely Response", "Complaints"]

timely_summary

,Timely Response,Complaints
0,Yes,60339
1,No,214


In [18]:
# timely response performance by product

timely_by_product = (
    df.groupby("Product")["Timely response?"]
    .value_counts()
    .unstack(fill_value=0)
)

timely_by_product["Total Complaints"] = timely_by_product.sum(axis=1)

timely_by_product["Timely Response Rate"] = (
    timely_by_product.get("Yes", 0)
    / timely_by_product["Total Complaints"]
    * 100
).round(2)

timely_by_product

Timely response?,No,Yes,Total Complaints,Timely Response Rate
Product,,,,
Credit card,13,4583,4596,99.72
"Money transfer, virtual currency, or money service",201,55756,55957,99.64


In [19]:
# what channels are customers using to report friction?

channel_summary = (
    df["Submitted via"]
    .value_counts()
    .reset_index()
)

channel_summary.columns = ["Channel", "Complaints"]

channel_summary

,Channel,Complaints
0,Web,59728
1,Referral,367
2,Phone,359
3,Postal mail,99


In [20]:
# Does the type of customer problem vary by complaint submission channel?

channel_issue = pd.crosstab(
    df["Submitted via"],
    df["Issue"]
)

channel_issue

Issue,Other service problem,Other transaction problem,Problem when making payments
Submitted via,,,
Phone,24,93,242
Postal mail,2,16,81
Referral,8,55,304
Web,2030,53729,3969


In [21]:
# looking at the monthly complaint volume

monthly_complaints = (
    df.set_index("Date received")
      .resample("MS")
      .size()
      .reset_index(name="Complaints")
)

monthly_complaints

,Date received,Complaints
0,2025-01-01 00:00:00+00:00,44236
1,2025-02-01 00:00:00+00:00,4234
2,2025-03-01 00:00:00+00:00,1641
3,2025-04-01 00:00:00+00:00,1135
4,2025-05-01 00:00:00+00:00,1263
5,2025-06-01 00:00:00+00:00,1007
6,2025-07-01 00:00:00+00:00,1007
7,2025-08-01 00:00:00+00:00,971
8,2025-09-01 00:00:00+00:00,870
9,2025-10-01 00:00:00+00:00,2341


In [22]:
# Investigating the January spike

january_daily = (
    df[df["Date received"].dt.month == 1]
    .set_index("Date received")
    .resample("D")
    .size()
    .reset_index(name="Complaints")
)

january_daily.sort_values("Complaints", ascending=False).head(15)

,Date received,Complaints
16,2025-01-17 00:00:00+00:00,13845
17,2025-01-18 00:00:00+00:00,8756
14,2025-01-15 00:00:00+00:00,4072
19,2025-01-20 00:00:00+00:00,4051
15,2025-01-16 00:00:00+00:00,3835
18,2025-01-19 00:00:00+00:00,2686
20,2025-01-21 00:00:00+00:00,1903
21,2025-01-22 00:00:00+00:00,979
22,2025-01-23 00:00:00+00:00,803
23,2025-01-24 00:00:00+00:00,562


In [23]:
# Further Investigating the January complaint spike

spike_period = df[
    (df["Date received"].dt.date >= pd.Timestamp("2025-01-15").date()) &
    (df["Date received"].dt.date <= pd.Timestamp("2025-01-21").date())
]

print(f"Complaints during Jan 15–21: {len(spike_period):,}")
print(f"Share of entire dataset: {len(spike_period) / len(df) * 100:.2f}%")

print("\nIssue distribution:")
print(
    spike_period["Issue"]
    .value_counts()
)

print("\nProduct distribution:")
print(
    spike_period["Product"]
    .value_counts()
)

Complaints during Jan 15–21: 39,148
Share of entire dataset: 64.65%

Issue distribution:
Issue
Other transaction problem       38013
Other service problem             945
Problem when making payments      190
Name: count, dtype: int64

Product distribution:
Product
Money transfer, virtual currency, or money service    38958
Credit card                                             190
Name: count, dtype: int64


In [24]:
# Identifying the spike driver

spike_company = (
    spike_period["Company"]
    .value_counts()
    .reset_index()
)

spike_company.columns = ["Company", "Complaints"]

spike_company.head(20)

,Company,Complaints
0,"Block, Inc.",25392
1,"Early Warning Services, LLC",13116
2,JPMORGAN CHASE & CO.,123
3,"BANK OF AMERICA, NATIONAL ASSOCIATION",94
4,WELLS FARGO & COMPANY,72
5,CAPITAL ONE FINANCIAL CORPORATION,41
6,NAVY FEDERAL CREDIT UNION,23
7,"Paypal Holdings, Inc",21
8,"CITIBANK, N.A.",18
9,SYNCHRONY FINANCIAL,16


In [25]:
spike_company["Share of Spike %"] = (
    spike_company["Complaints"]
    / len(spike_period)
    * 100
).round(2)

spike_company.head(20)

,Company,Complaints,Share of Spike %
0,"Block, Inc.",25392,64.86
1,"Early Warning Services, LLC",13116,33.50
2,JPMORGAN CHASE & CO.,123,0.31
3,"BANK OF AMERICA, NATIONAL ASSOCIATION",94,0.24
4,WELLS FARGO & COMPANY,72,0.18
5,CAPITAL ONE FINANCIAL CORPORATION,41,0.10
6,NAVY FEDERAL CREDIT UNION,23,0.06
7,"Paypal Holdings, Inc",21,0.05
8,"CITIBANK, N.A.",18,0.05
9,SYNCHRONY FINANCIAL,16,0.04


In [26]:
# The specific spike arear in the two dominant companies

top_spike_companies = [
    "Block, Inc.",
    "Early Warning Services, LLC"
]

company_subproduct = (
    spike_period[
        spike_period["Company"].isin(top_spike_companies)
    ]
    .groupby(["Company", "Sub-product"])
    .size()
    .reset_index(name="Complaints")
    .sort_values(
        ["Company", "Complaints"],
        ascending=[True, False]
    )
)

company_subproduct

,Company,Sub-product,Complaints
0,"Block, Inc.",Domestic (US) money transfer,25272
2,"Block, Inc.",Virtual currency,85
1,"Block, Inc.",International money transfer,35
3,"Early Warning Services, LLC",Domestic (US) money transfer,13031
5,"Early Warning Services, LLC",Virtual currency,55
4,"Early Warning Services, LLC",International money transfer,30


In [27]:
# Comparing the localized spike with the entire year

company_year = (
    df[df["Company"].isin(top_spike_companies)]
    .groupby("Company")
    .size()
    .reset_index(name="Full Year Complaints")
)

company_year

,Company,Full Year Complaints
0,"Block, Inc.",33588
1,"Early Warning Services, LLC",16750


In [28]:
spike_vs_year = company_year.merge(
    spike_company[
        spike_company["Company"].isin(top_spike_companies)
    ][["Company", "Complaints", "Share of Spike %"]],
    on="Company",
    how="left"
)

spike_vs_year.columns = [
    "Company",
    "Full Year Complaints",
    "Jan 15-21 Complaints",
    "Share of Total Spike %"
]

spike_vs_year["Spike Share of Company Year %"] = (
    spike_vs_year["Jan 15-21 Complaints"]
    / spike_vs_year["Full Year Complaints"]
    * 100
).round(2)

spike_vs_year

,Company,Full Year Complaints,Jan 15-21 Complaints,Share of Total Spike %,Spike Share of Company Year %
0,"Block, Inc.",33588,25392,64.86,75.6
1,"Early Warning Services, LLC",16750,13116,33.50,78.3


In [29]:
company_timing = (
    spike_period[
        spike_period["Company"].isin(top_spike_companies)
    ]
    .copy()
)

company_timing["Days to Company"] = (
    company_timing["Date sent to company"]
    - company_timing["Date received"]
).dt.total_seconds() / 86400

company_timing.groupby("Company")["Days to Company"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
Company,,,,,,,,
"Block, Inc.",25392.0,0.45,1.23,0.0,0.01,0.02,0.03,62.16
"Early Warning Services, LLC",13116.0,0.08,1.40,0.0,0.01,0.01,0.01,61.31


In [30]:
# CX Intelligence scorecard

cx_scorecard = (
    df.groupby(["Product", "Issue"])
    .agg(
        Complaints=("Complaint ID", "count"),
        Timely_Responses=("Timely response?", lambda x: (x == "Yes").sum()),
        Untimely_Responses=("Timely response?", lambda x: (x == "No").sum()),
        Monetary_Relief=("Company response to consumer", lambda x: (x == "Closed with monetary relief").sum()),
        Non_Monetary_Relief=("Company response to consumer", lambda x: (x == "Closed with non-monetary relief").sum())
    )
    .reset_index()
)

cx_scorecard["Complaint Share %"] = (
    cx_scorecard["Complaints"]
    / len(df)
    * 100
).round(2)

cx_scorecard["Timely Response Rate %"] = (
    cx_scorecard["Timely_Responses"]
    / cx_scorecard["Complaints"]
    * 100
).round(2)

cx_scorecard["Relief Rate %"] = (
    (
        cx_scorecard["Monetary_Relief"]
        + cx_scorecard["Non_Monetary_Relief"]
    )
    / cx_scorecard["Complaints"]
    * 100
).round(2)

cx_scorecard

,Product,Issue,Complaints,Timely_Responses,Untimely_Responses,Monetary_Relief,Non_Monetary_Relief,Complaint Share %,Timely Response Rate %,Relief Rate %
0,Credit card,Problem when making payments,4596,4583,13,802,413,7.59,99.72,26.44
1,"Money transfer, virtual currency, or money ser...",Other service problem,2064,2024,40,64,34,3.41,98.06,4.75
2,"Money transfer, virtual currency, or money ser...",Other transaction problem,53893,53732,161,471,162,89.00,99.70,1.17


In [31]:
cx_scorecard["Priority Tier"] = pd.cut(
    cx_scorecard["Complaint Share %"],
    bins=[-1, 5, 20, 100],
    labels=["Monitor", "High", "Critical"]
)

cx_scorecard.sort_values(
    "Complaints",
    ascending=False
)

,Product,Issue,Complaints,Timely_Responses,Untimely_Responses,Monetary_Relief,Non_Monetary_Relief,Complaint Share %,Timely Response Rate %,Relief Rate %,Priority Tier
2,"Money transfer, virtual currency, or money ser...",Other transaction problem,53893,53732,161,471,162,89.00,99.70,1.17,Critical
0,Credit card,Problem when making payments,4596,4583,13,802,413,7.59,99.72,26.44,High
1,"Money transfer, virtual currency, or money ser...",Other service problem,2064,2024,40,64,34,3.41,98.06,4.75,Monitor


In [32]:
transformation_opportunities = pd.DataFrame({
    "Customer Problem": [
        "Other transaction problem",
        "Problem when making payments",
        "Other service problem"
    ],
    
    "CX Signal": [
        "High concentration of transaction-related complaints",
        "Payment-related customer friction",
        "Service-related customer friction"
    ],
    
    "Potential Intervention": [
        "Improve transaction reliability, exception handling and proactive failure communication",
        "Simplify payment journeys, strengthen payment-status visibility and automate failure recovery",
        "Improve service workflows, self-service capability and escalation management"
    ],
    
    "Transformation Theme": [
        "Journey reliability & proactive service",
        "Digital journey optimisation & automation",
        "Service process redesign & self-service"
    ]
})

transformation_opportunities

,Customer Problem,CX Signal,Potential Intervention,Transformation Theme
0,Other transaction problem,High concentration of transaction-related comp...,"Improve transaction reliability, exception han...",Journey reliability & proactive service
1,Problem when making payments,Payment-related customer friction,"Simplify payment journeys, strengthen payment-...",Digital journey optimisation & automation
2,Other service problem,Service-related customer friction,"Improve service workflows, self-service capabi...",Service process redesign & self-service


In [33]:
df["Anomaly Period"] = (
    (df["Date received"] >= pd.Timestamp("2025-01-15", tz="UTC")) &
    (df["Date received"] <= pd.Timestamp("2025-01-21 23:59:59", tz="UTC"))
)

df["Anomaly Period"].value_counts()

Anomaly Period
True     39148
False    21405
Name: count, dtype: int64

In [34]:
monthly_trend = (
    df.assign(Month=df["Date received"].dt.to_period("M").astype(str))
      .groupby(["Month", "Anomaly Period"])
      .size()
      .reset_index(name="Complaints")
      .sort_values("Month")
)

monthly_trend

C:\Users\user\AppData\Local\Temp\ipykernel_28380\3001877799.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df.assign(Month=df["Date received"].dt.to_period("M").astype(str))


,Month,Anomaly Period,Complaints
0,2025-01,False,5088
1,2025-01,True,39148
2,2025-02,False,4234
3,2025-03,False,1641
4,2025-04,False,1135
5,2025-05,False,1263
6,2025-06,False,1007
7,2025-07,False,1007
8,2025-08,False,971
9,2025-09,False,870


In [35]:
# ============================================================
# PREPARE DATASETS FOR POWER BI
# ============================================================

from pathlib import Path

processed_folder = Path("../data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Clean main complaint dataset
# ------------------------------------------------------------

powerbi_df = df.copy()

# Remove timezone information for easier Power BI date handling
for column in ["Date received", "Date sent to company"]:
    powerbi_df[column] = powerbi_df[column].dt.tz_localize(None)

# Create useful analytical fields
powerbi_df["Month"] = powerbi_df["Date received"].dt.to_period("M").astype(str)

powerbi_df["Day of Week"] = (
    powerbi_df["Date received"]
    .dt.day_name()
)

powerbi_df["Anomaly Label"] = powerbi_df["Anomaly Period"].map({
    True: "Identified Spike",
    False: "Normal Period"
})

# Export
powerbi_df.to_csv(
    processed_folder / "complaints_clean.csv",
    index=False
)

# ------------------------------------------------------------
# 2. CX Scorecard
# ------------------------------------------------------------

cx_scorecard.to_csv(
    processed_folder / "cx_scorecard.csv",
    index=False
)

# ------------------------------------------------------------
# 3. Monthly trend
# ------------------------------------------------------------

monthly_trend.to_csv(
    processed_folder / "monthly_trend.csv",
    index=False
)

# ------------------------------------------------------------
# 4. Transformation opportunities
# ------------------------------------------------------------

transformation_opportunities.to_csv(
    processed_folder / "transformation_opportunities.csv",
    index=False
)

print("Processed datasets exported successfully.")
print()
print("Files created:")

for file in sorted(processed_folder.iterdir()):
    print(f"- {file.name}")

Processed datasets exported successfully.

Files created:
- complaints_clean.csv
- cx_scorecard.csv
- monthly_trend.csv
- transformation_opportunities.csv


# Executive CX Intelligence Findings

## 1. Customer friction is highly concentrated

The analysis identified three complaint categories within the selected population:

- Other transaction problem
- Problem when making payments
- Other service problem

Other transaction problems accounted for the overwhelming majority of complaints, indicating that transaction reliability represents the dominant customer-friction signal within this analytical population.

## 2. Response speed is not the same as customer experience quality

99.65% of complaints received a timely response.

However, high response timeliness does not necessarily indicate that the underlying customer journey is frictionless. Complaint volume remains an important signal of recurring customer problems even when organisations respond within the expected timeframe.

## 3. A significant complaint anomaly was identified

39,148 complaints were received between January 15 and January 21, representing 64.65% of the analytical population.

98.36% of this spike was concentrated in complaints involving Block, Inc. and Early Warning Services, LLC.

This should be treated as an anomaly requiring further operational investigation rather than evidence of a confirmed root cause.

## 4. CX transformation priorities

Based on the available evidence, three transformation opportunity areas were identified:

1. Transaction reliability and proactive service
2. Payment journey optimisation and automation
3. Service process redesign and self-service

These are intervention hypotheses that should be validated against internal operational, journey, customer and service data before implementation.